# Sprint 2: Descarga de images_004 y images_005

Agrega ~10k imágenes más al subset NIH para aumentar los casos de Infiltration.

**Tiempo estimado:** 40-80 min por tarball (~80-160 min total).
**Espacio necesario:** ~7.5 GB libres adicionales.
**Runtime:** CPU.

Al terminar, veremos si Infiltration tiene suficientes casos para
justificar 4 clases o si seguimos con 3.


## 1. Setup

In [ ]:
from google.colab import drive
try:
    drive.flush_and_unmount()
except:
    pass

drive.mount('/content/drive', force_remount=True)

import os, shutil, tarfile, time
from pathlib import Path

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
NIH_DIR = f'{PROJECT_ROOT}/data/raw/nih'
NIH_IMAGES_DIR = f'{NIH_DIR}/images'
BASE_URL = 'https://archive.org/download/ChestXray-NIHCC/ChestXray-NIHCC'

total, used, free = shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive libre: {free/(1024**3):.1f} GB')

# Drive a veces tarda en sincronizar carpetas grandes
# Reintentamos hasta 3 veces con pausa
for attempt in range(3):
    try:
        n_imgs = len([f for f in os.listdir(NIH_IMAGES_DIR) if f.endswith('.png')])
        print(f'Imágenes actuales: {n_imgs:,}')
        break
    except OSError as e:
        print(f'Intento {attempt+1}/3: Drive aún no sincroniza ({e}). Esperando 15s...')
        time.sleep(15)
else:
    # Si los 3 intentos fallan, usamos el CSV como proxy
    import pandas as pd
    df = pd.read_csv(f'{NIH_DIR}/Data_Entry_subset_local.csv')
    print(f'No se pudo leer la carpeta de imágenes directamente.')
    print(f'Usando CSV como referencia: {len(df):,} imágenes registradas.')
    n_imgs = len(df)


## 2. Descargar images_004 y images_005

In [ ]:
BASE_URL = 'https://archive.org/download/ChestXray-NIHCC/ChestXray-NIHCC'

def download_and_extract(filename: str) -> int:
    url = f'{BASE_URL}/{filename}'
    tar_path = f'{NIH_DIR}/{filename}'

    # --- Descarga ---
    if os.path.exists(tar_path) and os.path.getsize(tar_path) > 3_500_000_000 * 0.95:
        print(f'✓ {filename} ya existe ({os.path.getsize(tar_path)/(1024**3):.2f} GB). Saltando descarga.')
    else:
        print(f'\nDescargando {filename} (~3.7 GB)...')
        os.system(f'wget -c --progress=bar:force:noscroll -O "{tar_path}" "{url}"')
        size = os.path.getsize(tar_path) if os.path.exists(tar_path) else 0
        print(f'✓ Descargado ({size/(1024**3):.2f} GB)')

    # --- Verificar integridad ---
    try:
        with tarfile.open(tar_path, 'r:gz') as tar:
            n_files = sum(1 for m in tar.getmembers() if m.isfile())
        print(f'✓ Íntegro: {n_files} archivos')
    except Exception as e:
        print(f'✗ Corrupto: {e}')
        return 0

    # --- Extraer solo imágenes nuevas ---
    current_imgs = set(f for f in os.listdir(NIH_IMAGES_DIR) if f.endswith('.png'))
    print(f'Extrayendo imágenes nuevas de {filename}...')
    from tqdm import tqdm
    with tarfile.open(tar_path, 'r:gz') as tar:
        members = [m for m in tar.getmembers()
                   if m.isfile() and os.path.basename(m.name) not in current_imgs]
        for m in tqdm(members, desc=filename):
            tar.extract(m, path=NIH_DIR)

    new_total = len([f for f in os.listdir(NIH_IMAGES_DIR) if f.endswith('.png')])
    added = new_total - len(current_imgs)
    print(f'✓ +{added:,} imágenes nuevas | Total: {new_total:,}')
    return new_total

# Descargar y extraer ambos
print('=' * 60)
print('TARBALL 4')
print('=' * 60)
total_after_004 = download_and_extract('images_004.tar.gz')

print()
print('=' * 60)
print('TARBALL 5')
print('=' * 60)
total_after_005 = download_and_extract('images_005.tar.gz')


## 3. Actualizar el CSV local con las 5 tarballs

In [ ]:
import pandas as pd

df_full = pd.read_csv(f'{NIH_DIR}/Data_Entry_2017.csv')
all_imgs = set(f for f in os.listdir(NIH_IMAGES_DIR) if f.endswith('.png'))
df_subset = df_full[df_full['Image Index'].isin(all_imgs)].copy()
df_subset.to_csv(f'{NIH_DIR}/Data_Entry_subset_local.csv', index=False)

print(f'CSV actualizado: {len(df_subset):,} filas ({len(all_imgs):,} imágenes)')


## 4. Distribución de las 4 clases con 5 tarballs

In [ ]:
import pandas as pd

df = pd.read_csv(f'{NIH_DIR}/Data_Entry_subset_local.csv')

print('DISTRIBUCIÓN CON 5 TARBALLS (~25k imágenes):')
print('=' * 60)

classes = {
    'No Finding':    df['Finding Labels'] == 'No Finding',
    'Cardiomegaly':  df['Finding Labels'].str.contains('Cardiomegaly'),
    'Effusion':      df['Finding Labels'].str.contains('Effusion'),
    'Infiltration':  df['Finding Labels'].str.contains('Infiltration'),
}

for name, mask in classes.items():
    n = mask.sum()
    print(f'{name:<20}: {n:>6,}')

print()

# Estimar pacientes únicos por clase (lo que importa para el balance)
print('PACIENTES ÚNICOS POR CLASE:')
print('(esto define el techo del dataset balanceado)')
print('-' * 60)

# Aplicar jerarquía para asignar label a cada imagen
def assign_label(finding):
    if 'Infiltration' in finding: return 3
    if 'Effusion' in finding:     return 2
    if 'Cardiomegaly' in finding: return 1
    if finding == 'No Finding':   return 0
    return -1  # otros

df['label'] = df['Finding Labels'].apply(assign_label)
df_valid = df[df['label'] >= 0].copy()

# Paciente = patient_id con label máximo
patient_labels = df_valid.groupby('Patient ID')['label'].max()
for label, name in {0:'No Finding', 1:'Cardiomegaly', 2:'Effusion', 3:'Infiltration'}.items():
    n_patients = (patient_labels == label).sum()
    print(f'{name:<20}: {n_patients:>5,} pacientes únicos')

ceiling = min((patient_labels == l).sum() for l in [1, 2, 3])
n_nf = min((patient_labels == 0).sum(), ceiling * 2)
print()
print(f'Techo (clase más pequeña entre positivas): {ceiling:,} pacientes')
print(f'Dataset balanceado estimado:')
print(f'  {ceiling} x 3 positivas + {n_nf} No Finding = {ceiling*3 + n_nf:,} imágenes')
print()

# Decisión
if ceiling >= 80:
    print('✓ Suficiente para 4 clases (≥80 pacientes por clase).')
    print('  Recomendación: seguir con 4 clases.')
elif ceiling >= 40:
    print('⚠ Marginal para 4 clases (40-79 pacientes por clase).')
    print('  Recomendación: puedes intentar 4 clases con class weights.')
else:
    print('✗ Insuficiente para 4 clases (<40 pacientes).')
    print('  Recomendación: volver a 3 clases (NF + Cardio + Effusion).')


## 5. Siguiente paso

Según los números de la celda 4:
- Si **≥80 pacientes de Infiltration** → 4 clases viables, continuar Sprint 2 con 4 clases.
- Si **40-79 pacientes** → posible con class weights, decisión opcional.
- Si **<40 pacientes** → 3 clases (NF + Cardio + Effusion), Infiltration pasa a Sprint 3.

Vuelve al chat con la salida de la celda 4 y decidimos.
